# One-Click Flickr8k Paper Reproduction

Run this notebook with a GPU runtime. It clones the repo into `/content`, downloads the Flickr8k dataset automatically if not already present, and writes checkpoints/results to `/content/drive/MyDrive/Rohan_DL_Project_Outputs`.

No dataset upload required — the setup step downloads the public Flickr8k image and text archives (~1 GB) on first run.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Miwi343/Neural_Image_Caption_Generator.git'
BRANCH = ""  # Leave blank for the repository default branch.
REPO_DIR = Path('/content/Neural_Image_Caption_Generator')
OUTPUT_DIR = Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = ['git', 'clone', '--depth', '1']
if BRANCH:
    clone_cmd.extend(['--branch', BRANCH])
clone_cmd.extend([REPO_URL, str(REPO_DIR)])
subprocess.run(clone_cmd, check=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name in ('checkpoints', 'results'):
    target = OUTPUT_DIR / name
    target.mkdir(parents=True, exist_ok=True)
    local = REPO_DIR / name
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)
print('Outputs:', OUTPUT_DIR)


## Install Dependencies

In [ ]:
import torch

!python -m pip install -q -r requirements.txt
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## Prepare And Validate Flickr8k

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from config import VOCAB_SIZE
from utils import Vocabulary, validate_dataset_layout

DATA_ROOT = REPO_DIR / 'data' / 'flickr8k'
assert DATA_ROOT.exists(), f'Dataset folder missing from cloned repo: {DATA_ROOT}'

images_dir = DATA_ROOT / 'images'
dataset_present = (
    images_dir.exists()
    and (DATA_ROOT / 'captions.txt').exists()
    and len(list(images_dir.glob('*.jpg'))) > 100
)

if not dataset_present:
    print('Flickr8k images not found — downloading public release (~1 GB)...')
    subprocess.run([
        sys.executable,
        'scripts/colab_setup.py',
        '--repo_dir', str(REPO_DIR),
        '--data_root', 'data/flickr8k',
        '--download_public_flickr8k',
    ], check=True, cwd=str(REPO_DIR))
else:
    subprocess.run([
        sys.executable,
        'scripts/colab_setup.py',
        '--repo_dir', str(REPO_DIR),
        '--data_root', 'data/flickr8k',
        '--source_dir', str(DATA_ROOT),
        '--require_official_splits',
    ], check=True, cwd=str(REPO_DIR))

validate_dataset_layout(str(DATA_ROOT), strict_split_counts=True)

vocab_path = DATA_ROOT / 'vocab.json'
if vocab_path.exists():
    vocab = Vocabulary.load(str(vocab_path))
    if len(vocab) != VOCAB_SIZE:
        print(f'Removing incompatible vocab.json with size {len(vocab)}; train.py will rebuild {VOCAB_SIZE}.')
        vocab_path.unlink()

print('Flickr8k validated:', DATA_ROOT)


## Assert Paper Configuration

In [ ]:
import config

expected = {
    'EMBED_DIM': 512,
    'DECODER_DIM': 512,
    'ATTENTION_DIM': 512,
    'ENCODER_DIM': 512,
    'DROPOUT': 0.5,
    'VOCAB_SIZE': 10000,
    'LAMBDA': 1.0,
    'GRAD_CLIP': 5.0,
    'LEARNING_RATE': 4e-4,
    'BATCH_SIZE': 64,
    'NUM_EPOCHS': 50,
    'MAX_DECODE_LEN': 50,
}

bad = {name: (getattr(config, name), value) for name, value in expected.items() if getattr(config, name) != value}
assert not bad, f'Paper config mismatch: {bad}'
assert config.ENCODER_FINETUNE_EPOCH > config.NUM_EPOCHS, 'Encoder fine-tuning must stay disabled for the paper run.'
assert config.DATA_ROOT == 'data/flickr8k'
assert config.VOCAB_PATH == 'data/flickr8k/vocab.json'
print('Paper configuration verified.')


## Run Tests

In [ ]:
!pytest -q


## Train

In [ ]:
!python train.py


## Validate And Test Best Checkpoint

In [ ]:
!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split val   --beam_width 1   --batch_size 64   --results_out results/val_bleu.json

!python evaluate.py   --checkpoint checkpoints/best.pt   --data_root data/flickr8k   --vocab data/flickr8k/vocab.json   --split test   --beam_width 1   --batch_size 64   --results_out results/test_bleu.json


## Attention Visualizations

In [ ]:
!python visualize.py   --checkpoint checkpoints/best.pt   --vocab data/flickr8k/vocab.json   --data_root data/flickr8k   --split test   --num_examples 6   --output_dir results/attention_examples   --overlay_style paper   --dpi 200


In [ ]:
from IPython.display import Image, display
from pathlib import Path

for path in sorted(Path('results/attention_examples').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))


## Training Curve

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

log_path = Path('results/training_log.csv')
epochs, losses, bleu4 = [], [], []
with log_path.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row['epoch']))
        losses.append(float(row['train_loss']))
        bleu4.append(float(row['val_bleu4']) * 100.0)

fig, ax1 = plt.subplots(figsize=(8, 4), dpi=140)
ax1.plot(epochs, losses, color='tab:blue', marker='o', linewidth=1.5, label='Train loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train loss', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(epochs, bleu4, color='tab:green', marker='s', linewidth=1.5, label='Val BLEU-4')
ax2.set_ylabel('Val BLEU-4', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

fig.tight_layout()
curve_path = Path('results/training_curve.png')
fig.savefig(curve_path, bbox_inches='tight')
plt.show()
print('Saved', curve_path)


## Output Location

In [ ]:
from pathlib import Path

print('All persistent outputs are in:')
print(Path('/content/drive/MyDrive/Rohan_DL_Project_Outputs'))
print('
Expected files:')
for path in sorted(Path('results').glob('*')):
    print(path)
for path in sorted(Path('checkpoints').glob('*'))[:5]:
    print(path)
